# LLM Zoomcamp 2026 — Homework 4: Evaluation

Evaluating keyword, vector and hybrid search over the course lessons, using a
ground-truth set of questions and the Hit Rate / MRR metrics.

The pipeline is the same one from homework 2 (same chunks, same search
functions); here we add the vector and hybrid evaluation so we can compare the
three on numbers instead of intuition. We pin commit `8c1834d` so the 72 lesson
pages are identical for everyone.

## Setup

In [ ]:
# Install dependencies (run once)
%pip install -q openai pydantic python-dotenv pandas scikit-learn minsearch gitsource sentence-transformers tiktoken

In [ ]:
# The data generation in Q1 uses the course helpers
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
!wget -q -O rag_helper.py        {PREFIX}/01-agentic-rag/code/rag_helper.py
!wget -q -O evaluation_utils.py  {PREFIX}/04-evaluation/code/evaluation_utils.py

In [ ]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
len(documents)  # -> 72

## Q1. Generating questions

For the first 3 lesson pages we ask the LLM to write 5 questions answered by
each page, then look at the average number of **input** tokens across the 3
calls.

In [ ]:
import os, json
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import llm_structured

load_dotenv()
client = OpenAI()  # reads OPENAI_API_KEY from .env


class Questions(BaseModel):
    questions: list[str]


data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()


def build_user_prompt(doc):
    return json.dumps({"filename": doc["filename"], "content": doc["content"]})


first_three = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]
docs3 = sorted(
    [d for d in documents if d["filename"] in first_three],
    key=lambda d: first_three.index(d["filename"]),
)

input_tokens = []
for d in docs3:
    user_prompt = build_user_prompt(d)
    parsed, usage = llm_structured(client, data_gen_instructions, user_prompt, Questions)
    input_tokens.append(usage.input_tokens)
    print(d["filename"], "-> input tokens:", usage.input_tokens)

avg = sum(input_tokens) / len(input_tokens)
print("Average input tokens:", round(avg))

In [ ]:
# No OpenAI key? Estimate the input tokens locally (same prompt, no API call).
# The number stays in the same order of magnitude, which is all Q1 needs.
def estimate_tokens(text):
    try:
        import tiktoken
        return len(tiktoken.get_encoding("o200k_base").encode(text))
    except Exception:
        return round(len(text) / 4)  # rough fallback if the encoding can't be fetched

est = [estimate_tokens(data_gen_instructions) + estimate_tokens(build_user_prompt(d)) for d in docs3]
print("Estimated avg input tokens:", round(sum(est) / len(est)), "-> closest option: 1400")

## Searching the chunks

Same chunks as homework 2, and the same text / vector / hybrid search built on
top of them.

In [ ]:
import pandas as pd

ground_truth = pd.read_csv(f"{PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv").to_dict(orient="records")
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks), len(ground_truth)  # -> 295, 360

In [ ]:
from minsearch import Index, VectorSearch
from sentence_transformers import SentenceTransformer

# Text index (keyword search), keyed on filename
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

# Vector index, same embedding model as the module
model = SentenceTransformer("all-MiniLM-L6-v2")
X = model.encode([c["content"] for c in chunks], show_progress_bar=True)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

def vector_search(query, num_results=5):
    return vector_index.search(model.encode(query), num_results=num_results)

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores, docs = {}, {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Q2. First result with text search

In [ ]:
q = ground_truth[0]["question"]
text_search(q)[0]["filename"]  # -> 01-agentic-rag/lessons/03-rag.md

## Q3. First result with vector search

In [ ]:
vector_search(q)[0]["filename"]  # -> 01-agentic-rag/lessons/01-intro.md

## Evaluation metrics

A result counts as a hit when a returned chunk's `filename` matches the
question's `filename`.

In [ ]:
from tqdm.auto import tqdm

def compute_relevance(search_function, gt):
    results = search_function(gt["question"])
    return [d["filename"] == gt["filename"] for d in results]

def hit_rate(relevance_total):
    return sum(1 for line in relevance_total if True in line) / len(relevance_total)

def mrr(relevance_total):
    total = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank]:
                total += 1 / (rank + 1)
                break
    return total / len(relevance_total)

def evaluate(search_function, ground_truth):
    relevance = [compute_relevance(search_function, gt) for gt in tqdm(ground_truth)]
    return {"hit_rate": hit_rate(relevance), "mrr": mrr(relevance)}

## Q4. Evaluating text search

In [ ]:
evaluate(text_search, ground_truth)["hit_rate"]  # -> ~0.76

## Q5. Evaluating vector search

In [ ]:
evaluate(vector_search, ground_truth)["mrr"]  # -> pick the closest option

## Q6. Tuning hybrid search

A smaller `k` sharpens the gap between top positions. We measure MRR for a few
values and pick the best (smallest `k` on a tie).

In [ ]:
results = {}
for k in [1, 50, 100, 200]:
    results[k] = evaluate(lambda query: hybrid_search(query, k=k), ground_truth)["mrr"]

results, max(results, key=results.get)

## Summary of answers

In [ ]:
print("Q1:", round(sum(input_tokens)/len(input_tokens)) if input_tokens else "run cell above", "(~1400)")
print("Q2:", text_search(q)[0]["filename"])
print("Q3:", vector_search(q)[0]["filename"])
print("Q4 hit rate:", round(evaluate(text_search, ground_truth)["hit_rate"], 4))
print("Q5 vector MRR:", round(evaluate(vector_search, ground_truth)["mrr"], 4))
print("Q6 best k:", max(results, key=results.get))